# 한국어 로컬 LLM 비교 — 회의록 근거 답변 생성

`main.py` 채팅(`/api/v1/chat`)의 **답변 생성 단계**에 쓸 로컬 LLM을 고르기 위한 비교 노트북입니다.
프로젝트의 `.py` 파일을 import하지 않고 모든 코드를 셀 안에 담았습니다.

**비교 방식**
1. 질문 8개(대표 이슈 7개 + 근거에 없는 내용을 묻는 미끼 질문 1개)마다 PostgreSQL에서 회의록 근거 6건을 **한 번만** 검색해 저장합니다.
   모든 모델이 **같은 근거·같은 프롬프트**를 받으므로 차이는 모델에서만 생깁니다.
2. 프롬프트는 `api/services.py` `RadarApiService.chat()`의 시스템 프롬프트와 근거 블록 형식을 그대로 씁니다.
3. 모델마다 워밍업 후 8개 질문에 답하게 하고, 끝나면 모델을 메모리에서 내립니다(다음 모델이 GPU를 온전히 쓰도록).
4. 답변을 자동 지표로 채점합니다: 응답 시간·생성 속도, 한글 비율, `[근거 N]` 인용, **근거에 없는 숫자·날짜·의회명**(검증),
   미끼 질문에서 "근거 부족"이라고 답했는지, `<think>` 노출 여부.

**사전 조건**: pgvector DB(`council_arctic_raw_v1`), 로컬 Ollama와 비교할 모델들, HuggingFace 캐시의 임베딩 모델.
설정은 같은 폴더의 `.env`를 읽고, 없으면 제출 폴더 루트의 `../.env`를 씁니다.
결과는 제출 폴더의 `data/llm_compare/`에 저장되며, 다시 실행하면 이미 끝난 (모델, 질문)은 건너뜁니다.

> 자동 지표는 사람 평가를 대신하지 않습니다. 최종 선택 전에 6번 셀에서 답변 원문을 직접 읽어 보세요.

## 1. 설정

In [ ]:
from __future__ import annotations

import json
import os
import re
import shlex
import statistics
import time
import urllib.request
from collections import defaultdict
from datetime import date, datetime
from pathlib import Path
from typing import Any
from urllib.parse import urlparse

from IPython.display import Markdown, display

BASE_DIR = Path.cwd()          # 이 노트북이 있는 폴더에서 실행한다
OUT_DIR = BASE_DIR.parent / "data" / "llm_compare"   # 제출 폴더의 data/llm_compare
OUT_DIR.mkdir(parents=True, exist_ok=True)


def load_env_file(path: Path) -> int:
    """KEY=VALUE 형식의 .env를 읽는다. 셸에서 이미 지정한 값은 덮어쓰지 않는다."""
    if not path.is_file():
        return 0
    loaded = 0
    for raw in path.read_text(encoding="utf-8-sig").splitlines():
        line = raw.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.removeprefix("export ").split("=", 1)
        key = key.strip()
        if key in os.environ:
            continue
        try:
            value = " ".join(shlex.split(value, comments=True, posix=True))
        except ValueError:
            value = value.strip().strip("\"'")
        os.environ[key] = value
        loaded += 1
    return loaded


for env_path in (BASE_DIR / ".env", BASE_DIR.parent / ".env"):
    if env_path.is_file():
        print(f"{env_path} 에서 {load_env_file(env_path)}개 설정 로드")
        break
else:
    print(".env를 찾지 못했습니다. 환경 변수만 사용합니다.")

SETTINGS = {
    "dsn": os.environ.get("PGVECTOR_DSN", ""),
    "table": os.environ.get("COUNCIL_PG_TABLE", "council_arctic_raw_v1"),
    "ef_search": max(1, int(os.environ.get("COUNCIL_PG_EF_SEARCH", "200"))),
    "embedding_model": os.environ.get("COUNCIL_EMBEDDING_MODEL", "dragonkue/snowflake-arctic-embed-l-v2.0-ko"),
    "embedding_revision": os.environ.get("COUNCIL_EMBEDDING_REVISION", "55ec6e9358a56d56af759bc8372e970caf8c305f"),
    "max_seq_length": int(os.environ.get("COUNCIL_MAX_SEQ_LENGTH", "2048")),
    "device": os.environ.get("COUNCIL_DEVICE", "auto"),
    "model_cache": os.environ.get("COUNCIL_MODEL_CACHE"),
    "allow_download": os.environ.get("COUNCIL_ALLOW_DOWNLOAD", "0") == "1",
    "ollama_url": os.environ.get("OLLAMA_URL", "http://127.0.0.1:11434"),
}
if not SETTINGS["dsn"]:
    raise RuntimeError("PGVECTOR_DSN이 없습니다 (.env 확인)")

# 비교할 모델 (Ollama 이름). 첫 줄이 기존 main.py 기본 모델이다.
MODELS = [
    "exaone3.5:7.8b",                                                          # LG AI연구원 (기존 기본값)
    "hf.co/mykor/A.X-4.0-Light-gguf:Q4_K_M",                                   # SK텔레콤 A.X 4.0 Light
    "hf.co/mradermacher/kanana-1.5-8b-instruct-2505-GGUF:Q4_K_M",              # 카카오 Kanana 1.5
    "hf.co/DevQuasar/trillionlabs.Tri-7B-GGUF:Q4_K_M",                         # 트릴리온랩스 Tri-7B
    "hf.co/rippertnt/HyperCLOVAX-SEED-Text-Instruct-1.5B-Q4_K_M-GGUF:Q4_K_M",  # 네이버 HyperCLOVA X SEED 1.5B
    "hf.co/LGAI-EXAONE/EXAONE-4.0-1.2B-GGUF:Q4_K_M",                           # LG EXAONE 4.0 1.2B
]
TOP_K = 6              # 질문당 근거 수 (ChatRequest 기본값)
EVIDENCE_CHARS = 1800  # 근거 1건당 프롬프트에 넣는 글자 수 (services.py와 동일)
TIMEOUT = 600          # 모델 응답 대기 시간(초)
# Ollama 기본 컨텍스트(4096 토큰)는 근거 6건 프롬프트(약 4,100토큰)보다 작아 앞부분이 잘릴 수 있다.
# 모든 모델에 같은 값을 주어 잘림 없이 비교한다. None이면 Ollama 기본값을 쓴다.
NUM_CTX: int | None = 8192
print(f"모델 {len(MODELS)}개 · 근거 {TOP_K}건 · num_ctx {NUM_CTX} · 결과 폴더 {OUT_DIR}")

## 2. 비교 질문
대표 이슈 7개에서 하나씩 뽑은 실제 정책 질문과, 회의록에 있을 수 없는 내용을 묻는 **미끼 질문**(`hallucination_bait`)입니다.
미끼 질문에는 "근거가 부족하다"고 답하는 모델이 좋은 모델입니다.

In [ ]:
QUESTIONS = [
    {"id": "birth_rate", "q": "출산율 저하에 대응하기 위해 지방의회에서 어떤 정책이 논의됐나요?"},
    {"id": "population_decline", "q": "지방소멸과 인구감소 대응으로 의회에서 제안된 사업을 알려줘"},
    {"id": "youth_jobs", "q": "청년 일자리와 지역정착 지원에 대해 의원들이 지적한 문제점은?"},
    {"id": "jeonse_fraud", "q": "전세사기 피해자 지원 대책으로 어떤 내용이 나왔어?"},
    {"id": "regional_healthcare", "q": "지역 필수의료 공백 문제에 대해 회의록에서 어떤 대안이 언급됐나요?"},
    {"id": "school_violence", "q": "학교폭력 피해 학생 보호를 위해 논의된 내용은?"},
    {"id": "flood_safety", "q": "집중호우 침수 피해 예방 대책으로 어떤 사업 예산이 논의됐는지 금액과 함께 알려줘"},
    {"id": "hallucination_bait", "q": "2025년 화성 이주민 지원 조례를 처음 통과시킨 의회와 예산 규모는?"},
]
for item in QUESTIONS:
    print(f"{item['id']:20} {item['q']}")

## 3. Ollama 확인
회의록이 외부로 나가지 않도록 localhost Ollama만 허용합니다. 설치되지 않은 모델은 비교에서 빼고 받는 명령을 알려 줍니다.

In [ ]:
LOCAL_HOSTS = {"localhost", "127.0.0.1", "::1"}
OLLAMA = SETTINGS["ollama_url"].rstrip("/")
if (urlparse(OLLAMA).hostname or "").lower() not in LOCAL_HOSTS:
    raise RuntimeError("보안 정책상 localhost Ollama만 사용할 수 있습니다")


def ollama_post(path: str, body: dict[str, Any], timeout: int = TIMEOUT) -> dict[str, Any]:
    request = urllib.request.Request(OLLAMA + path, data=json.dumps(body, ensure_ascii=False).encode("utf-8"),
                                     headers={"Content-Type": "application/json"}, method="POST")
    with urllib.request.urlopen(request, timeout=timeout) as response:
        return json.loads(response.read().decode("utf-8"))


with urllib.request.urlopen(OLLAMA + "/api/tags", timeout=5) as response:
    installed = {m["name"] for m in json.loads(response.read().decode("utf-8")).get("models") or []}
missing_models = [m for m in MODELS if m not in installed]
MODELS_TO_RUN = [m for m in MODELS if m in installed]
print(f"Ollama 연결됨 · 비교할 모델 {len(MODELS_TO_RUN)}/{len(MODELS)}개")
for model in missing_models:
    print(f"  설치 안 됨 → 건너뜀:  ollama pull {model}")

## 4. 질문별 회의록 근거 (모든 모델 공통)
질문을 임베딩해 pgvector에서 가장 가까운 청크 `TOP_K`개를 가져와 `data/llm_compare/evidence_top6.json`에 저장합니다.
LLM 비교에는 모델 간 **같은 근거**면 충분하므로 dense(HNSW) 검색만 씁니다.
(전문검색까지 섞는 하이브리드 검색은 인덱스가 없어 질문당 수 분이 걸리기 때문입니다. DB는 읽기만 합니다.)

In [ ]:
import psycopg
import torch
from pgvector.psycopg import register_vector
from psycopg.rows import dict_row
from sentence_transformers import SentenceTransformer

TABLE = SETTINGS["table"]
EVIDENCE_PATH = OUT_DIR / f"evidence_top{TOP_K}.json"
evidence: dict[str, list[dict[str, Any]]] = (
    json.loads(EVIDENCE_PATH.read_text(encoding="utf-8")) if EVIDENCE_PATH.is_file() else {})
need = [item for item in QUESTIONS if item["id"] not in evidence]

if need:
    device = SETTINGS["device"]
    if device == "auto":
        device = "cuda" if torch.cuda.is_available() else "cpu"
    kwargs: dict[str, Any] = {"device": device, "local_files_only": not SETTINGS["allow_download"],
                              "revision": SETTINGS["embedding_revision"]}
    if SETTINGS["model_cache"]:
        kwargs["cache_folder"] = str(Path(SETTINGS["model_cache"]).expanduser().resolve())
    encoder = SentenceTransformer(SETTINGS["embedding_model"], **kwargs)
    encoder.max_seq_length = SETTINGS["max_seq_length"]
    prompt = {"prompt_name": "query"} if "query" in getattr(encoder, "prompts", {}) else {}
    with psycopg.connect(SETTINGS["dsn"], row_factory=dict_row) as conn:
        register_vector(conn)
        conn.execute("SELECT set_config('hnsw.ef_search', %s, false)", (str(SETTINGS["ef_search"]),))
        for item in need:
            vector = encoder.encode([item["q"]], normalize_embeddings=True, **prompt)[0].astype("float32").tolist()
            rows = conn.execute(f"""
                SELECT chunk_id, council, meeting_date, meeting_name, document,
                       1 - (embedding <=> %s::vector) AS similarity
                FROM {TABLE} WHERE embedding IS NOT NULL
                ORDER BY embedding <=> %s::vector, chunk_id LIMIT %s""", [vector, vector, TOP_K]).fetchall()
            evidence[item["id"]] = [{
                "chunk_id": str(r["chunk_id"]),
                "metadata": {"council": r["council"], "meeting_name": r["meeting_name"],
                             "meeting_date": r["meeting_date"].isoformat() if r["meeting_date"] else None},
                "similarity": round(float(r["similarity"]), 4),
                "text": r["document"] or "",
            } for r in rows]
            EVIDENCE_PATH.write_text(json.dumps(evidence, ensure_ascii=False, indent=1), encoding="utf-8")
            print(f"근거 검색 {item['id']}: {len(rows)}건")
else:
    print(f"저장된 근거 사용: {EVIDENCE_PATH}")

for item in QUESTIONS:
    rows = evidence[item["id"]]
    print(f"{item['id']:20} " + " | ".join(f"{r['metadata']['council']} {r['metadata']['meeting_date']} ({r['similarity']:.2f})"
                                           for r in rows[:3]) + " …")

## 5. 프롬프트·채점 함수
- **프롬프트**: `api/services.py`의 시스템 프롬프트와 `[근거 N] 의회 | 날짜` 블록 형식 그대로.
- **검증**(`verify_answer`, 원본 `agent/query_pipeline.py`와 같은 규칙): 답변 속 숫자·날짜·의회명이 근거와 질문에 있는지 대조합니다.
  연도(2000~2100)와 10 이하 정수는 제외하고, 반올림·백분율 표현은 허용합니다. 범위를 벗어난 `[근거 N]` 번호도 실패로 봅니다.

In [ ]:
SYSTEM_PROMPT = (  # api/services.py RadarApiService.chat()과 동일
    "지방의회 회의록 근거를 사용해 한국어로 답하세요. 근거에 없는 사실이나 숫자는 만들지 말고, "
    "각 사실 뒤에 [근거 N]을 표시하세요. 근거가 부족하면 그렇게 말하세요."
)


def build_messages(question: str, rows: list[dict[str, Any]]) -> tuple[list[dict[str, str]], str]:
    blocks = [f"[근거 {i}] {r['metadata'].get('council', '미상')} | {r['metadata'].get('meeting_date', '날짜 미상')}\n"
              f"{r['text'][:EVIDENCE_CHARS]}" for i, r in enumerate(rows, start=1)]
    evidence_text = "\n\n".join(blocks)
    return [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"질문: {question}\n\nPostgreSQL 검색 근거:\n" + evidence_text}], evidence_text


# --- 의회 이름 표기 (원본 external_policy_sources.REGION, regional_stats.SHORT)
REGION = {
    "서울특별시의회": ["서울특별시"], "부산광역시의회": ["부산광역시"], "대구광역시의회": ["대구광역시"],
    "인천광역시의회": ["인천광역시"], "광주광역시의회": ["광주광역시"], "대전광역시의회": ["대전광역시"],
    "울산광역시의회": ["울산광역시"], "세종특별자치시의회": ["세종특별자치시"], "경기도의회": ["경기도"],
    "강원특별자치도의회": ["강원특별자치도", "강원도"], "충청북도의회": ["충청북도"], "충청남도의회": ["충청남도"],
    "전북특별자치도의회": ["전북특별자치도", "전라북도"], "전라남도의회": ["전라남도"], "경상북도의회": ["경상북도"],
    "경상남도의회": ["경상남도"], "제주특별자치도의회": ["제주특별자치도"],
}
SHORT_REGION = {"서울": "서울특별시의회", "부산": "부산광역시의회", "대구": "대구광역시의회", "인천": "인천광역시의회",
                "광주": "광주광역시의회", "대전": "대전광역시의회", "울산": "울산광역시의회", "세종": "세종특별자치시의회",
                "경기": "경기도의회", "강원": "강원특별자치도의회", "충북": "충청북도의회", "충남": "충청남도의회",
                "전북": "전북특별자치도의회", "전남": "전라남도의회", "경북": "경상북도의회", "경남": "경상남도의회",
                "제주": "제주특별자치도의회"}
REGION_FORMS: dict[str, str] = {}
for council, forms in REGION.items():
    REGION_FORMS[council] = council
    REGION_FORMS[council.removesuffix("의회")] = council
    for form in forms:
        REGION_FORMS[form] = council
for short, council in SHORT_REGION.items():
    REGION_FORMS[short] = council
    REGION_FORMS[short + ("도" if council.endswith("도의회") else "시")] = council   # 경북도, 부산시 …
REGION_FORMS.update({"전라북도": "전북특별자치도의회", "강원도": "강원특별자치도의회", "제주도": "제주특별자치도의회"})

NUMBER_PATTERN = re.compile(r"\d+(?:,\d{3})*(?:\.\d+)?")
KOREAN_DATE = re.compile(r"(\d{4})\s*년\s*(\d{1,2})\s*월(?:\s*(\d{1,2})\s*일)?")


def _numbers(text: str) -> list[float]:
    return [float(value.replace(",", "")) for value in NUMBER_PATTERN.findall(text)]


def _number_supported(value: float, allowed: list[float]) -> bool:
    """반올림(0.531→0.53)·백분율(0.531→53.1%) 표현까지 허용"""
    for known in allowed:
        for candidate in (known, known * 100):
            if abs(candidate - value) <= max(0.006, abs(candidate) * 0.01):
                return True
    return False


def _iso_dates(text: str) -> str:
    """'2023년 5월 25일' → '2023-05-25' (날짜 대조를 위해)"""
    return KOREAN_DATE.sub(lambda m: f"{m[1]}-{int(m[2]):02d}" + (f"-{int(m[3]):02d}" if m[3] else ""), text)


def verify_answer(answer: str, evidence_text: str, question: str, evidence_count: int) -> dict[str, Any]:
    """답변 속 숫자·날짜·의회명·근거 번호가 근거·질문에 있는지 대조 (연도·10 이하 정수는 제외)"""
    allowed = evidence_text + " " + _iso_dates(question)
    allowed_numbers = _numbers(allowed)
    bad_citations = sorted({f"[근거 {n}]" for n in map(int, re.findall(r"\[근거\s*(\d+)\]", answer))
                            if not 1 <= n <= evidence_count})
    cleaned = re.sub(r"\[(근거|외부)[^\]]*\]", " ", _iso_dates(answer))
    cleaned_for_numbers = re.sub(r"\d{4}-\d{2}(-\d{2})?", " ", cleaned)
    unknown_numbers = sorted({f"{v:g}" for v in _numbers(cleaned_for_numbers)
                              if not (v.is_integer() and (v <= 10 or 2000 <= v <= 2100))
                              and not _number_supported(v, allowed_numbers)})
    mentioned = {REGION_FORMS[f] for f in REGION_FORMS if len(f) >= 3 and f in cleaned}
    allowed_councils = {REGION_FORMS[f] for f in REGION_FORMS if len(f) >= 3 and f in allowed}
    date_pattern = re.compile(r"\d{4}-\d{2}(?:-\d{2})?")
    unknown_dates = sorted(set(date_pattern.findall(cleaned)) - set(date_pattern.findall(allowed)))
    unknown_councils = sorted(mentioned - allowed_councils)
    return {"passed": not unknown_numbers and not unknown_councils and not unknown_dates and not bad_citations,
            "unsupported": unknown_numbers[:10] + unknown_dates[:5] + bad_citations + unknown_councils}


INSUFFICIENT = re.compile(r"근거[가 ]*(부족|없)|찾을 수 없|확인되지 않|언급되지 않|포함되어 있지 않")


def score(answer: str, question: str, evidence_text: str, evidence_count: int) -> dict[str, Any]:
    hangul = len(re.findall(r"[가-힣]", answer))
    letters = len(re.findall(r"[A-Za-z가-힣一-鿿]", answer)) or 1
    check = verify_answer(answer, evidence_text, question, evidence_count)
    return {
        "chars": len(answer),
        "hangul_ratio": round(hangul / letters, 3),
        "citation_count": len(re.findall(r"\[근거\s*\d+\]", answer)),
        "distinct_citations": len({int(n) for n in re.findall(r"\[근거\s*(\d+)\]", answer)}),
        "says_insufficient": bool(INSUFFICIENT.search(answer)),
        "think_leak": "<think>" in answer or "</think>" in answer,
        "verify_passed": check["passed"],
        "unsupported": check["unsupported"],
    }


# 동작 확인: 근거에 없는 금액·의회가 들어간 답은 실패해야 한다
demo = verify_answer("부산광역시의회에서 37억 원을 편성했습니다 [근거 1]", "[근거 1] 서울특별시의회 | 2024-05-01\n예산 12억 원", "질문", 1)
print("검증 예시:", demo)

## 6. 모델별 답변 생성
모델마다 ① 짧은 워밍업(모델 로딩 시간이 응답 시간에 섞이지 않게) → ② 질문 8개 답변 → ③ 메모리에서 내리기 순서로 실행합니다.
결과는 한 줄씩 `data/llm_compare/results.jsonl`에 저장되어, 중간에 멈춰도 다시 실행하면 이어서 진행합니다.
모델 6개 기준 GPU에서 10~20분 정도 걸립니다.

In [ ]:
RESULTS_PATH = OUT_DIR / "results.jsonl"
OPTIONS = {"num_ctx": NUM_CTX} if NUM_CTX else {}


def load_results() -> list[dict[str, Any]]:
    if not RESULTS_PATH.is_file():
        return []
    return [json.loads(line) for line in RESULTS_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]


done = {(r["model"], r["question_id"]) for r in load_results()}
with RESULTS_PATH.open("a", encoding="utf-8") as out:
    for model in MODELS_TO_RUN:
        todo = [item for item in QUESTIONS if (model, item["id"]) not in done]
        if not todo:
            print(f"[{model}] 이미 완료")
            continue
        try:
            ollama_post("/api/chat", {"model": model, "stream": False, "options": OPTIONS,
                                      "messages": [{"role": "user", "content": "안녕"}]})
        except Exception as error:
            print(f"[{model}] 워밍업 실패 → 건너뜀: {error}")
            continue
        for item in todo:
            rows = evidence[item["id"]]
            messages, evidence_text = build_messages(item["q"], rows)
            started = time.perf_counter()
            try:
                payload, error = ollama_post("/api/chat", {"model": model, "stream": False,
                                                           "messages": messages, "options": OPTIONS}), None
            except Exception as exc:
                payload, error = {}, f"{type(exc).__name__}: {exc}"
            wall = time.perf_counter() - started
            answer = str((payload.get("message") or {}).get("content") or "").strip()
            eval_count = payload.get("eval_count") or 0
            eval_seconds = (payload.get("eval_duration") or 0) / 1e9
            record = {
                "model": model, "question_id": item["id"], "question": item["q"],
                "wall_seconds": round(wall, 2), "prompt_tokens": payload.get("prompt_eval_count"),
                "output_tokens": eval_count,
                "tokens_per_second": round(eval_count / eval_seconds, 1) if eval_seconds else None,
                "error": error, "answer": answer, **score(answer, item["q"], evidence_text, len(rows)),
            }
            out.write(json.dumps(record, ensure_ascii=False) + "\n")
            out.flush()
            print(f"[{model}] {item['id']}: {wall:.1f}s · 인용 {record['citation_count']} · "
                  + ("검증 통과" if record["verify_passed"] else f"검증 실패 {record['unsupported']}")
                  + (f" · 오류 {error}" if error else ""))
        try:   # 다음 모델이 GPU 메모리를 온전히 쓰도록 내린다
            ollama_post("/api/generate", {"model": model, "keep_alive": 0}, timeout=60)
        except Exception:
            pass
print("완료:", len(load_results()), "건")

## 7. 결과 요약
| 지표 | 의미 | 좋은 방향 |
|---|---|---|
| 평균 응답(s) | 질문 하나에 답하는 데 걸린 시간 (워밍업 제외) | 짧을수록 |
| 생성 속도(tok/s) | 초당 생성 토큰 | 클수록 |
| 한글 비율 | 답변 글자 중 한글 비율 (영어·한자 섞임 확인) | 1.0에 가까울수록 |
| 인용 포함 문항 · 평균 인용 수 | `[근거 N]` 표시 여부·개수 | 많을수록 |
| 검증 통과 | 근거에 없는 숫자·날짜·의회명이 없는 답변 수 | 많을수록 |
| 미끼 질문 '근거 부족' | 회의록에 없는 질문에 지어내지 않고 근거 부족이라 답했는지 | 예 |

In [ ]:
results = load_results()
by_model: dict[str, list[dict[str, Any]]] = defaultdict(list)
for row in results:
    by_model[row["model"]].append(row)

lines = ["| 모델 | 문항 | 오류 | 평균 응답(s) | 생성 속도(tok/s) | 평균 길이(자) | 한글 비율 | 인용 포함 문항 | 평균 인용 수 | 검증 통과 | 미끼 질문 '근거 부족' | think 노출 |",
         "|---|---|---|---|---|---|---|---|---|---|---|---|"]
for model in [m for m in MODELS if m in by_model]:
    items = by_model[model]
    ok = [r for r in items if not r["error"] and r["answer"]]
    bait = [r for r in items if r["question_id"] == "hallucination_bait"]

    def avg(key: str) -> float:
        values = [r[key] for r in ok if r[key] is not None]
        return statistics.mean(values) if values else 0.0

    lines.append(
        f"| {model} | {len(items)} | {len(items) - len(ok)} | {avg('wall_seconds'):.1f} | {avg('tokens_per_second'):.1f} | "
        f"{avg('chars'):.0f} | {avg('hangul_ratio'):.2f} | {sum(r['citation_count'] > 0 for r in ok)}/{len(ok)} | "
        f"{avg('citation_count'):.1f} | {sum(r['verify_passed'] for r in ok)}/{len(ok)} | "
        f"{'예' if bait and bait[0]['says_insufficient'] else '아니오'} | {sum(r['think_leak'] for r in ok)} |")
REPORT = "\n".join(lines)
(OUT_DIR / "report.md").write_text(REPORT + "\n", encoding="utf-8")
display(Markdown(REPORT))

## 8. 답변 원문 보기
`SHOW_QUESTION`에 질문 id를 넣으면 그 질문에 대한 모델별 답변을 나란히 보여 줍니다.
전체 답변은 `data/llm_compare/answers.md`에도 저장합니다.

In [ ]:
SHOW_QUESTION = "hallucination_bait"   # QUESTIONS의 id 중 하나


def answer_block(row: dict[str, Any]) -> str:
    verdict = "검증 통과" if row["verify_passed"] else "검증 실패: " + ", ".join(row["unsupported"])
    return (f"### {row['model']}\n*{row['wall_seconds']}s · 인용 {row['citation_count']} · {verdict}*\n\n"
            + (row["answer"] or f"(오류: {row['error']})") + "\n")


sections = []
for item in QUESTIONS:
    group = [r for r in results if r["question_id"] == item["id"]]
    sections.append(f"## {item['id']}: {item['q']}\n\n" + "\n".join(answer_block(r) for r in group))
(OUT_DIR / "answers.md").write_text("\n".join(sections), encoding="utf-8")

question = next(item for item in QUESTIONS if item["id"] == SHOW_QUESTION)
display(Markdown(f"## {question['id']}: {question['q']}\n\n"
                 + "\n".join(answer_block(r) for r in results if r["question_id"] == SHOW_QUESTION)))

## 9. 해석 시 유의점
- 질문 8개의 소규모 비교라 문항 하나의 차이가 크게 보입니다. 순위보다 **오류 유형**(지어낸 숫자, 인용 누락, 외국어 섞임)을 보세요.
- 검증은 숫자·날짜·의회명만 대조합니다. 근거와 다른 **내용**을 말해도 숫자가 없으면 통과할 수 있습니다.
- 응답 시간은 GPU·양자화(Q4_K_M)·답변 길이에 좌우됩니다. 같은 PC에서 같은 설정으로 잰 상대값으로만 비교하세요.
- 노트북의 채팅은 근거 외에 이슈 분석·예측·외부 데이터 요약도 함께 넣습니다. 이 비교는 **회의록 근거만** 넣은 조건입니다.
- 모델을 바꾸려면 제출 폴더 `.env`의 `COUNCIL_AGENT_MODEL`을 바꾸면 됩니다.